# Выборочные статистики и центральная предельная теорема

Учебный пример к **занятию 5**: как по одной выборке оценивать статистики генеральной совокупности и почему такие оценки сами по себе случайны.

Данные — скорость ветра `wvel` на метеостанции **Малые Кармакулы** (таблица подготовлена на втором занятии, `Lect02_EDA_demo.ipynb`).

План работы:

1. взять из всего ряда скорости ветра небольшую подвыборку (100 значений) и посмотреть на её распределение;
2. методом **bootstrap** (сэмплирование с возвращением) получить много новых выборок и проследить, как ведёт себя **выборочное среднее**;
3. убедиться, что распределение средних сужается с ростом размера подвыборки, — это проявление **центральной предельной теоремы (ЦПТ)**.

## Полезные ссылки
- <a href="https://pandas.pydata.org/">Документация pandas</a>
- <a href="https://numpy.org/doc/stable/">Документация NumPy</a>
- <a href="https://matplotlib.org/">Документация Matplotlib</a>
- <a href="https://seaborn.pydata.org/">Документация Seaborn</a>
- [Центральная предельная теорема](https://en.wikipedia.org/wiki/Central_limit_theorem)
- <a href="https://en.wikipedia.org/wiki/Bootstrapping_(statistics)">Bootstrap (сэмплирование с возвращением)</a>

## Дополнительные материалы
- Ноутбук `Lect05b_Bootstrap_demo_synthetic_data.ipynb` — bootstrap для оценки неопределённости параметров модели
- Ноутбук `Lect05c_CV_real_data.ipynb` — кросс-валидация на реальных данных
- [Series.sample](https://pandas.pydata.org/docs/reference/api/pandas.Series.sample.html) — сэмплирование с возвращением (`replace=True`)
- [Учебник «An Introduction to Statistical Learning»](https://www.statlearning.com/)


## Импорт библиотек

Подключаем числовые и табличные инструменты, а также графику Matplotlib и Seaborn. Две последние строки задают разрешение сохраняемых и отображаемых картинок.

**Инструменты:** [NumPy](https://numpy.org/doc/stable/), [pandas](https://pandas.pydata.org/docs/), [matplotlib.pyplot](https://matplotlib.org/stable/api/pyplot_summary.html), [seaborn](https://seaborn.pydata.org/), [настройки matplotlib через rcParams](https://matplotlib.org/stable/api/matplotlib_configuration_api.html#matplotlib.rcParams).


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['savefig.dpi'] = 150
plt.rcParams['figure.dpi'] = 150

## Подготовка данных

И снова здравствуйте, Малые Кармакулы

Читаем таблицу, подготовленную на втором занятии (`Lect02_EDA_demo.ipynb`): колонка `Date` сразу делается индексом временного ряда, чтобы дальше было удобно брать выборки по датам.

Затем выбираем, что именно будем анализировать. По умолчанию (раскомментированная строка) из всего ряда скорости ветра `wvel` берётся **случайная подвыборка из 100 значений**; `random_state=42` фиксирует выбор, чтобы результат воспроизводился при повторных запусках. Если вместо этого раскомментировать первую строку, в анализ пойдёт весь ряд целиком — полезно сравнить, как изменятся выводы.

**Инструменты:** [pandas.read_csv](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html) (параметры `parse_dates` и `index_col`), [Series.dropna](https://pandas.pydata.org/docs/reference/api/pandas.Series.dropna.html), [Series.sample](https://pandas.pydata.org/docs/reference/api/pandas.Series.sample.html).


In [ ]:
df = pd.read_csv('EDA_demo_data/karmakuly_processed.csv', parse_dates=['Date'], index_col='Date')

# Все данные
#data2stat = df['wvel'].dropna()

# Выберем только 100 случайных элементов
data2stat = df['wvel'].dropna().sample(100, random_state=42)


## Как выглядит распределение скорости ветра в подвыборке

Строим гистограмму выбранных 100 значений `wvel`: по горизонтальной оси — скорость ветра, по вертикальной — число наблюдений, попавших в интервал (корзину). Распределение заметно скошено вправо: слабых ветров много, сильных — мало. Именно из этого распределения дальше будем многократно сэмплировать.

**Инструменты:** [Series.plot.hist](https://pandas.pydata.org/docs/reference/api/pandas.Series.plot.hist.html), [matplotlib.axes.Axes.set_xlabel](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.set_xlabel.html), [Axes.grid](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.grid.html).


In [ ]:
ax = data2stat.plot.hist (bins=30, color='steelblue', ec='black', linewidth=0.5)

ax.set_xlabel ('Скорость ветра wvel, м/с')
ax.set_ylabel ('Число наблюдений')
ax.set_title ('Распределение скорости ветра в подвыборке (n = 100)')
ax.grid (axis='y', alpha=0.3)


## Применим bootstrap для формирования выборок и анализа выборочных средних

**Bootstrap** — это сэмплирование из имеющейся выборки **с возвращением**: каждый элемент берётся случайно, может попасть в новую выборку несколько раз или не попасть вовсе. Так из одной выборки получают много «псевдовыборок» и смотрят, насколько устойчива интересующая статистика.

В ячейке ниже из подвыборки в 100 значений строится `n = 1000` bootstrap-выборок и рисуются три гистограммы:

1. **`1000 элементов`** — значения `wvel` в одной bootstrap-выборке: форма распределения повторяет исходную;
2. **`1000 средних из 10`** — выборочные средние по выборкам размера 10;
3. **`1000 средних из 100`** — выборочные средние по выборкам размера 100.

Видно, что разброс средних заметно меньше разброса самих значений и уменьшается с ростом размера выборки: стандартное отклонение среднего падает как $1/\sqrt{n}$. Это и есть **центральная предельная теорема** — распределение выборочного среднего стремится к нормальному и сужается при увеличении $n$. Список `subsample_sizes` можно менять (например, на `[5, 50]`), чтобы увидеть это своими глазами.

**Инструменты:** [DataFrame.sample](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sample.html), [pandas.concat](https://pandas.pydata.org/docs/reference/api/pandas.concat.html), [seaborn.FacetGrid](https://seaborn.pydata.org/generated/seaborn.FacetGrid.html), [FacetGrid.map](https://seaborn.pydata.org/generated/seaborn.FacetGrid.map.html), [FacetGrid.set_axis_labels](https://seaborn.pydata.org/generated/seaborn.FacetGrid.set_axis_labels.html).


In [ ]:
n = 1000

subsample_sizes = [10, 100]
#subsample_sizes = [5, 50]

# Одна bootstrap-выборка из n значений
sample_data = pd.DataFrame({'wvel':data2stat.sample(n, replace=True, random_state=42),
                            'type':'%d элементов'%n})
results = [sample_data]

# По n выборочных средних для каждого размера подвыборки
for sn in subsample_sizes:
    results.append(pd.DataFrame({'wvel':[data2stat.sample(sn, replace=True).mean() for _ in range (n)],
                                  'type':'%d средних из %d'%(n, sn)}))

results = pd.concat (results)

g = sns.FacetGrid (results, col = 'type', height = 4, aspect = 1)
g.map (plt.hist, 'wvel', bins=30, color='steelblue', ec='black', linewidth=0.5)
g.set_axis_labels ('Скорость ветра wvel, м/с', 'Число наблюдений')
g.set_titles (col_template = '{col_name}')
g.figure.suptitle ('Bootstrap: значения wvel и распределения выборочных средних', y=1.02)
